In [1]:
"""
Design Document Generator using OpenAI + Python
================================================
Generates a professional technical design document (Word .docx) from a subject/topic
using the OpenAI API. Saves output as a formatted Word document.

Requirements:
    pip install openai python-docx

Usage:
    python generate_design_doc.py --subject "E-Commerce Recommendation Engine"
    python generate_design_doc.py --subject "Microservices Auth System" --type HLD
    python generate_design_doc.py  # interactive prompt
"""

import argparse
import json
import os
import subprocess
import sys
import tempfile
from datetime import date



In [2]:
# ── OpenAI client setup ──────────────────────────────────────────────────────
try:
    from openai import OpenAI
except ImportError:
    print("❌  openai package not found. Run: pip install openai")
    sys.exit(1)

In [3]:
# ── Prompt templates ─────────────────────────────────────────────────────────
SYSTEM_PROMPT = """You are a senior software architect who writes clear, structured,
professional technical design documents. Return ONLY valid JSON — no markdown fences,
no commentary before or after. Follow the exact schema provided."""

DOC_TYPES = {
    "HLD": "High-Level Design (HLD)",
    "LLD": "Low-Level Design (LLD)",
    "SDD": "Software Design Document (SDD)",
    "TDD": "Technical Design Document (TDD)",
    "PRD": "Product Requirements Document (PRD)",
}


def build_user_prompt(subject: str, doc_type: str) -> str:
    return f"""Create a detailed {DOC_TYPES.get(doc_type, doc_type)} for: "{subject}"

Return EXACTLY this JSON structure (all fields required):
{{
  "title": "<document title>",
  "version": "1.0",
  "date": "{date.today().isoformat()}",
  "authors": ["Your Name"],
  "doc_type": "{DOC_TYPES.get(doc_type, doc_type)}",
  "executive_summary": "<2-3 sentence overview>",
  "objectives": ["<objective 1>", "<objective 2>", "<objective 3>"],
  "scope": {{
    "in_scope": ["<item 1>", "<item 2>", "<item 3>"],
    "out_of_scope": ["<item 1>", "<item 2>"]
  }},
  "architecture_overview": "<paragraph describing the system architecture>",
  "components": [
    {{
      "name": "<Component Name>",
      "description": "<what it does>",
      "responsibilities": ["<resp 1>", "<resp 2>"],
      "technology": "<tech stack>"
    }}
  ],
  "data_design": {{
    "overview": "<data model overview>",
    "key_entities": ["<Entity1>", "<Entity2>", "<Entity3>"],
    "storage": "<database/storage technology>"
  }},
  "api_design": {{
    "style": "<REST/GraphQL/gRPC/etc>",
    "endpoints": [
      {{"method": "GET",  "path": "/example",        "description": "<what it does>"}},
      {{"method": "POST", "path": "/example",        "description": "<what it does>"}},
      {{"method": "PUT",  "path": "/example/{{id}}", "description": "<what it does>"}}
    ]
  }},
  "security": {{
    "authentication": "<auth mechanism>",
    "authorization": "<authz approach>",
    "considerations": ["<security point 1>", "<security point 2>"]
  }},
  "scalability": "<paragraph on scaling strategy>",
  "non_functional_requirements": [
    {{"category": "Performance",    "requirement": "<requirement>"}},
    {{"category": "Availability",   "requirement": "<requirement>"}},
    {{"category": "Reliability",    "requirement": "<requirement>"}},
    {{"category": "Maintainability","requirement": "<requirement>"}}
  ],
  "risks": [
    {{"risk": "<risk description>", "mitigation": "<mitigation strategy>"}},
    {{"risk": "<risk description>", "mitigation": "<mitigation strategy>"}}
  ],
  "timeline": [
    {{"phase": "Phase 1 – Discovery",     "duration": "2 weeks", "deliverable": "<deliverable>"}},
    {{"phase": "Phase 2 – Development",   "duration": "6 weeks", "deliverable": "<deliverable>"}},
    {{"phase": "Phase 3 – Testing",       "duration": "2 weeks", "deliverable": "<deliverable>"}},
    {{"phase": "Phase 4 – Deployment",    "duration": "1 week",  "deliverable": "<deliverable>"}}
  ],
  "conclusion": "<closing paragraph summarising the document>"
}}"""

In [4]:
# ── OpenAI call ───────────────────────────────────────────────────────────────
def generate_doc_content(subject: str, doc_type: str, api_key: str) -> dict:
    """Call OpenAI and return parsed JSON design document content."""
    client = OpenAI(api_key=api_key)
    print(f"🤖  Calling OpenAI (gpt-4o) for subject: '{subject}' …")

    response = client.chat.completions.create(
        model="gpt-4o",
        temperature=0.4,
        response_format={"type": "json_object"},
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": build_user_prompt(subject, doc_type)},
        ],
    )
    raw = response.choices[0].message.content
    return json.loads(raw)

In [5]:
# ── Word document builder ─────────────────────────────────────────────────────
JS_TEMPLATE = r"""
const {
  Document, Packer, Paragraph, TextRun, Table, TableRow, TableCell,
  HeadingLevel, AlignmentType, BorderStyle, WidthType, ShadingType,
  LevelFormat, PageNumber, Footer, Header
} = require('docx');
const fs = require('fs');
const data = {DATA_JSON};
"""

In [6]:
// -- colour palette -----------------------------------------------------------
const BLUE   = "1F4E79";
const LBLUE  = "2E75B6";
const THEAD  = "D5E8F0";
const WHITE  = "FFFFFF";
const LGRAY  = "F2F2F2";
const DTEXT  = "2C2C2C";
const MTEXT  = "444444";
const ACCENT = "C00000";

SyntaxError: invalid syntax (288437827.py, line 1)

In [ ]:
// ── helpers ───────────────────────────────────────────────────────────────────
const cell = (text, opts = {{}}) => new TableCell({{
  width: {{ size: opts.width || 4680, type: WidthType.DXA }},
  shading: {{ fill: opts.fill || WHITE, type: ShadingType.CLEAR }},
  borders: {{
    top:    {{ style: BorderStyle.SINGLE, size: 1, color: "CCCCCC" }},
    bottom: {{ style: BorderStyle.SINGLE, size: 1, color: "CCCCCC" }},
    left:   {{ style: BorderStyle.SINGLE, size: 1, color: "CCCCCC" }},
    right:  {{ style: BorderStyle.SINGLE, size: 1, color: "CCCCCC" }},
  }},
  margins: {{ top: 80, bottom: 80, left: 120, right: 120 }},
  children: [new Paragraph({{
    children: [new TextRun({{ text: String(text), bold: opts.bold || false,
      color: opts.color || DTEXT, size: opts.size || 20 }})]
  }})]
}});

const h1 = (text) => new Paragraph({{
  heading: HeadingLevel.HEADING_1,
  spacing: {{ before: 360, after: 160 }},
  border: {{ bottom: {{ style: BorderStyle.SINGLE, size: 8, color: LBLUE, space: 1 }} }},
  children: [new TextRun({{ text, bold: true, color: BLUE, size: 32, font: "Arial" }})]
}});

const h2 = (text) => new Paragraph({{
  heading: HeadingLevel.HEADING_2,
  spacing: {{ before: 240, after: 120 }},
  children: [new TextRun({{ text, bold: true, color: LBLUE, size: 26, font: "Arial" }})]
}});

const body = (text) => new Paragraph({{
  spacing: {{ after: 160 }},
  children: [new TextRun({{ text: String(text), color: MTEXT, size: 20 }})]
}});

const bullet = (text) => new Paragraph({{
  numbering: {{ reference: "bullets", level: 0 }},
  spacing: {{ after: 80 }},
  children: [new TextRun({{ text: String(text), color: MTEXT, size: 20 }})]
}});

const space = () => new Paragraph({{ children: [new TextRun("")], spacing: {{ after: 120 }} }});

In [ ]:
// ── cover page ────────────────────────────────────────────────────────────────
const coverSection = [
  new Paragraph({{
    alignment: AlignmentType.CENTER,
    spacing: {{ before: 1440, after: 400 }},
    children: [new TextRun({{ text: data.doc_type.toUpperCase(), bold: true,
      color: WHITE, size: 24, highlight: "none" }})]
  }}).constructor === Paragraph
    ? new Paragraph({{
        alignment: AlignmentType.CENTER,
        spacing: {{ before: 1440, after: 400 }},
        children: [new TextRun({{ text: "◆  " + data.doc_type.toUpperCase() + "  ◆",
          bold: true, color: LBLUE, size: 28, font: "Arial" }})]
      }})
    : new Paragraph({{ children: [new TextRun("")] }}),

  new Paragraph({{
    alignment: AlignmentType.CENTER,
    spacing: {{ before: 0, after: 560 }},
    children: [new TextRun({{ text: data.title, bold: true, color: BLUE,
      size: 56, font: "Arial" }})]
  }}),

  new Paragraph({{
    alignment: AlignmentType.CENTER,
    spacing: {{ after: 160 }},
    children: [new TextRun({{ text: "Version " + data.version + "  |  " + data.date,
      color: MTEXT, size: 22 }})]
  }}),

  new Paragraph({{
    alignment: AlignmentType.CENTER,
    spacing: {{ after: 80 }},
    children: [new TextRun({{ text: "Author(s): " + data.authors.join(", "),
      color: MTEXT, size: 22 }})]
  }}),

  new Paragraph({{ pageBreakBefore: true, children: [new TextRun("")] }}),
];

In [ ]:
// ── build children array ──────────────────────────────────────────────────────
const children = [
  ...coverSection,

  // 1. Executive Summary
  h1("1. Executive Summary"),
  body(data.executive_summary),
  space(),

  // 2. Objectives
  h1("2. Objectives"),
  ...data.objectives.map(o => bullet(o)),
  space(),

  // 3. Scope
  h1("3. Scope"),
  h2("3.1 In-Scope"),
  ...data.scope.in_scope.map(s => bullet(s)),
  h2("3.2 Out-of-Scope"),
  ...data.scope.out_of_scope.map(s => bullet(s)),
  space(),

  // 4. Architecture Overview
  h1("4. Architecture Overview"),
  body(data.architecture_overview),
  space(),

  // 5. Components
  h1("5. System Components"),
  ...data.components.flatMap((c, i) => [
    h2(\`5.\${i+1} \${c.name}\`),
    body("Description: " + c.description),
    body("Technology: " + c.technology),
    body("Responsibilities:"),
    ...c.responsibilities.map(r => bullet(r)),
    space(),
  ]),

  // 6. Data Design
  h1("6. Data Design"),
  body(data.data_design.overview),
  body("Storage: " + data.data_design.storage),
  h2("Key Entities"),
  ...data.data_design.key_entities.map(e => bullet(e)),
  space(),

  // 7. API Design
  h1("7. API Design"),
  body("Style: " + data.api_design.style),
  space(),
  new Table({{
    width: {{ size: 9360, type: WidthType.DXA }},
    columnWidths: [1200, 3000, 5160],
    rows: [
      new TableRow({{
        tableHeader: true,
        children: [
          cell("Method",      {{ width: 1200, fill: BLUE,  bold: true, color: WHITE }}),
          cell("Endpoint",    {{ width: 3000, fill: BLUE,  bold: true, color: WHITE }}),
          cell("Description", {{ width: 5160, fill: BLUE,  bold: true, color: WHITE }}),
        ]
      }}),
      ...data.api_design.endpoints.map((ep, idx) =>
        new TableRow({{
          children: [
            cell(ep.method,      {{ width: 1200, fill: idx%2===0 ? WHITE : LGRAY }}),
            cell(ep.path,        {{ width: 3000, fill: idx%2===0 ? WHITE : LGRAY }}),
            cell(ep.description, {{ width: 5160, fill: idx%2===0 ? WHITE : LGRAY }}),
          ]
        }})
      )
    ]
  }}),
  space(),

  // 8. Security
  h1("8. Security"),
  body("Authentication: " + data.security.authentication),
  body("Authorization:  " + data.security.authorization),
  ...data.security.considerations.map(s => bullet(s)),
  space(),

  // 9. Scalability
  h1("9. Scalability"),
  body(data.scalability),
  space(),

  // 10. Non-Functional Requirements
  h1("10. Non-Functional Requirements"),
  new Table({{
    width: {{ size: 9360, type: WidthType.DXA }},
    columnWidths: [2500, 6860],
    rows: [
      new TableRow({{
        tableHeader: true,
        children: [
          cell("Category",    {{ width: 2500, fill: BLUE, bold: true, color: WHITE }}),
          cell("Requirement", {{ width: 6860, fill: BLUE, bold: true, color: WHITE }}),
        ]
      }}),
      ...data.non_functional_requirements.map((nfr, idx) =>
        new TableRow({{
          children: [
            cell(nfr.category,    {{ width: 2500, fill: idx%2===0 ? WHITE : LGRAY, bold: true }}),
            cell(nfr.requirement, {{ width: 6860, fill: idx%2===0 ? WHITE : LGRAY }}),
          ]
        }})
      )
    ]
  }}),
  space(),

  // 11. Risks
  h1("11. Risks & Mitigations"),
  new Table({{
    width: {{ size: 9360, type: WidthType.DXA }},
    columnWidths: [4680, 4680],
    rows: [
      new TableRow({{
        tableHeader: true,
        children: [
          cell("Risk",       {{ width: 4680, fill: BLUE, bold: true, color: WHITE }}),
          cell("Mitigation", {{ width: 4680, fill: BLUE, bold: true, color: WHITE }}),
        ]
      }}),
      ...data.risks.map((r, idx) =>
        new TableRow({{
          children: [
            cell(r.risk,       {{ width: 4680, fill: idx%2===0 ? WHITE : LGRAY }}),
            cell(r.mitigation, {{ width: 4680, fill: idx%2===0 ? WHITE : LGRAY }}),
          ]
        }})
      )
    ]
  }}),
  space(),

  // 12. Timeline
  h1("12. Project Timeline"),
  new Table({{
    width: {{ size: 9360, type: WidthType.DXA }},
    columnWidths: [3500, 1860, 4000],
    rows: [
      new TableRow({{
        tableHeader: true,
        children: [
          cell("Phase",       {{ width: 3500, fill: BLUE, bold: true, color: WHITE }}),
          cell("Duration",    {{ width: 1860, fill: BLUE, bold: true, color: WHITE }}),
          cell("Deliverable", {{ width: 4000, fill: BLUE, bold: true, color: WHITE }}),
        ]
      }}),
      ...data.timeline.map((t, idx) =>
        new TableRow({{
          children: [
            cell(t.phase,       {{ width: 3500, fill: idx%2===0 ? WHITE : LGRAY }}),
            cell(t.duration,    {{ width: 1860, fill: idx%2===0 ? WHITE : LGRAY }}),
            cell(t.deliverable, {{ width: 4000, fill: idx%2===0 ? WHITE : LGRAY }}),
          ]
        }})
      )
    ]
  }}),
  space(),

  // 13. Conclusion
  h1("13. Conclusion"),
  body(data.conclusion),
];

In [ ]:
// ── assemble document ─────────────────────────────────────────────────────────
const doc = new Document({{
  numbering: {{
    config: [{{
      reference: "bullets",
      levels: [{{ level: 0, format: LevelFormat.BULLET, text: "•",
        alignment: AlignmentType.LEFT,
        style: {{ paragraph: {{ indent: {{ left: 720, hanging: 360 }} }} }} }}]
    }}]
  }},
  styles: {{
    default: {{
      document: {{ run: {{ font: "Arial", size: 20, color: DTEXT }} }}
    }},
    paragraphStyles: [
      {{ id: "Heading1", name: "Heading 1", basedOn: "Normal", next: "Normal",
         run: {{ size: 32, bold: true, font: "Arial" }},
         paragraph: {{ spacing: {{ before: 360, after: 160 }}, outlineLevel: 0 }} }},
      {{ id: "Heading2", name: "Heading 2", basedOn: "Normal", next: "Normal",
         run: {{ size: 26, bold: true, font: "Arial" }},
         paragraph: {{ spacing: {{ before: 240, after: 120 }}, outlineLevel: 1 }} }},
    ]
  }},
  sections: [{{
    properties: {{
      page: {{
        size: {{ width: 12240, height: 15840 }},
        margin: {{ top: 1080, right: 1080, bottom: 1080, left: 1080 }}
      }}
    }},
    footers: {{
      default: new Footer({{
        children: [new Paragraph({{
          alignment: AlignmentType.CENTER,
          children: [
            new TextRun({{ text: data.title + "  |  Page ", color: "888888", size: 16 }}),
            new TextRun({{ children: [PageNumber.CURRENT], color: "888888", size: 16 }}),
            new TextRun({{ text: " of ", color: "888888", size: 16 }}),
            new TextRun({{ children: [PageNumber.TOTAL_PAGES], color: "888888", size: 16 }}),
          ]
        }})]
      }})
    }},
    children
  }}]
}});

Packer.toBuffer(doc).then(buf => {{
  fs.writeFileSync(process.argv[2], buf);
  console.log("OK");
}}).catch(err => {{
  console.error(err.message);
  process.exit(1);
}});
"""


def build_docx(content: dict, out_path: str) -> None:
    """Render the design document as a formatted .docx via Node.js + docx-js."""
    js_code = JS_TEMPLATE.replace("{DATA_JSON}", json.dumps(content, ensure_ascii=False))

    with tempfile.NamedTemporaryFile(suffix=".js", mode="w", delete=False) as f:
        f.write(js_code)
        js_path = f.name

    try:
        result = subprocess.run(
            ["node", js_path, out_path],
            capture_output=True, text=True, timeout=60
        )
        if result.returncode != 0 or "OK" not in result.stdout:
            raise RuntimeError(result.stderr or result.stdout)
        print(f"✅  Word document saved → {out_path}")
    finally:
        os.unlink(js_path)

In [ ]:
# ── main ──────────────────────────────────────────────────────────────────────
def main():
    parser = argparse.ArgumentParser(
        description="Generate a Design Document (.docx) using OpenAI"
    )
    parser.add_argument("--subject", "-s", type=str,
                        help="Topic/subject for the design document")
    parser.add_argument("--type", "-t", default="HLD",
                        choices=list(DOC_TYPES.keys()),
                        help="Document type (default: HLD)")
    parser.add_argument("--output", "-o", type=str, default=None,
                        help="Output .docx file path (default: auto-named)")
    parser.add_argument("--api-key", type=str, default=None,
                        help="OpenAI API key (or set OPENAI_API_KEY env var)")
    args = parser.parse_args()

    # ── get API key ──────────────────────────────────────────────────────────
    api_key = args.api_key or os.environ.get("OPENAI_API_KEY", "")
    if not api_key:
        api_key = input("🔑  Enter your OpenAI API key: ").strip()
    if not api_key:
        print("❌  No API key provided. Exiting.")
        sys.exit(1)

    # ── get subject ──────────────────────────────────────────────────────────
    subject = args.subject
    if not subject:
        subject = input("📝  Enter the document subject/topic: ").strip()
    if not subject:
        print("❌  No subject provided. Exiting.")
        sys.exit(1)

    # ── output file path ─────────────────────────────────────────────────────
    safe_name = "".join(c if c.isalnum() else "_" for c in subject)[:50]
    out_path = args.output or f"{safe_name}_{args.type}.docx"

    # ── generate content ─────────────────────────────────────────────────────
    content = generate_doc_content(subject, args.type, api_key)
    print(f"✅  Content generated: '{content.get('title', subject)}'")

    # ── build word document ──────────────────────────────────────────────────
    build_docx(content, out_path)
    print(f"\n📄  Done! Open '{out_path}' to view your design document.")


if __name__ == "__main__":
    main()